# Actor-Critic（AC / A2C / A3C）重点笔记（含容易忽略的实现要点）

> 关键词：**Actor + Critic**、**Advantage / TD error**、**Bootstrap**、**Bias–Variance Tradeoff**、**n-step**、**Stop-Gradient**、**Advantage Normalization**、**Entropy Bonus**、**On-policy 数据新鲜度**

---

## 1. AC 的核心：比 REINFORCE 多了什么？解决什么？
- **多了 Critic（Value Network）**：学习 \(V_\phi(s)\) 作为 baseline
- **主要解决**：REINFORCE 方差太高（训练抖、样本效率差）
- **代价**：bootstrap + value approximation 带来 bias（critic 不准会带偏 actor）

---

## 2. Actor 更新（Policy Gradient with Advantage）

### 2.1 常见形式
\[
L_{\text{actor}}(\theta) = -\mathbb{E}[\log \pi_\theta(a_t|s_t)\, A_t]
\]

### 2.2 一步 Advantage（TD error）
\[
A_t \approx \delta_t = r_{t+1} + \gamma (1-d_t) V_\phi(s_{t+1}) - V_\phi(s_t)
\]

---

## 3. Critic 更新（Value Regression）

### 3.1 一步 TD target
\[
y_t = r_{t+1} + \gamma (1-d_t) V_\phi(s_{t+1})
\]

### 3.2 MSE loss
\[
L_{\text{critic}}(\phi) = \mathbb{E}\big[(V_\phi(s_t) - y_t)^2\big]
\]

---

## 4. n-step return（A2C/A3C 常用）

\[
y_t^{(n)}=\sum_{k=0}^{n-1}\gamma^k r_{t+1+k} + \gamma^n V_\phi(s_{t+n})
\]

关系：
- n 越大更接近 MC：bias 更小、方差更大
- n 越小更接近 TD(0)：方差更小、bias 更大
- TD(\(\lambda\)) ≈ 把不同 n-step return 做指数加权混合

---

## 5. A2C vs A3C（一句话）
- **A3C**：多 worker 异步采样/异步推梯度（吞吐高、实现复杂、梯度更噪）
- **A2C**：多 worker 同步采样/同步更新（GPU 友好、实现简单、更稳定）

---

# 6. 容易忽略但对理解/实现极其重要的点（必背清单）

## 6.1 Actor 更新必须 stop-gradient Advantage
- 你希望 actor 只根据优势信号更新策略，而不是“通过 critic 的梯度走捷径”
- 实现：`adv = stop_grad(adv)` / `adv = adv.detach()`

## 6.2 Critic 的 target 也通常 stop-gradient
- TD target 里的 \(V(s')\) 一般不回传梯度（否则“自己追自己”，不稳）
- 实现：`target = r + gamma * stop_grad(V(s_next))`

## 6.3 Actor/Critic 的学习率与更新比很敏感
- critic 太弱：优势估计噪声大 → actor 乱跑
- critic 太强/过拟合：优势偏 → actor 被带偏
- 常见：critic 学得稍“更充分”，并设置 value loss 系数 \(c_v\)

## 6.4 Advantage normalization 是隐形稳定器
- 实践常用：`adv = (adv - mean) / (std + eps)`
- 作用：让更新尺度稳定，训练显著更稳

## 6.5 共享 backbone vs 分离网络：结构性选择
- 共享：省参数、学习更快，但梯度可能互相干扰
- 分离：更稳，但更耗参数/算力

## 6.6 On-policy 数据“新鲜度”很关键
- rollout 太长或对同一批数据训练太多 epoch，会变得“半 off-policy”而不稳
- 这也是 PPO 用 clipping / KL 约束来允许多 epoch 的原因之一

## 6.7 连续动作必须管好方差 \(\sigma\) / 熵
- \(\sigma\) 过快变小 ⇒ 探索塌陷 ⇒ 容易卡局部最优
- 常用 entropy bonus：
\[
L = L_{\text{actor}} + c_v L_{\text{critic}} - c_e \mathcal H(\pi_\theta)
\]

---

# 7. 最常见的总 loss（A2C/A3C 风格）
\[
L(\theta,\phi) = 
-\mathbb{E}[\log \pi_\theta(a_t|s_t)\, A_t]
+ c_v\, \mathbb{E}[(V_\phi(s_t)-y_t)^2]
- c_e\, \mathbb{E}[\mathcal H(\pi_\theta(\cdot|s_t))]
\]

---

## 8. 记忆卡片（5 行）
- AC = policy gradient + value baseline（降方差）
- \(A_t \approx \delta_t = r+\gamma V(s')-V(s)\)
- bootstrap 降方差但引入 critic bias
- n-step/λ 都是在调 bias–variance
- 实现成败关键：stop-grad、adv norm、entropy、on-policy 新鲜度

---

*生成时间：2026-01-13 03:45:44*